Here’s a clean refresher note.

## End-to-end from your Windows PowerShell to GCP VM / Nginx app

### 1) Raw TCP reachability

**What it proves:** your laptop can reach the VM on a specific port.

**Example test**

```powershell
Test-NetConnection 136.114.99.52 -Port 80

output:
ComputerName     : 136.114.99.52
RemoteAddress    : 136.114.99.52
RemotePort       : 80
InterfaceAlias   : Wi-Fi
SourceAddress    : 192.168.2.178
TcpTestSucceeded : True

Test-NetConnection 136.114.99.52 -Port 443

output:
ComputerName     : 136.114.99.52
RemoteAddress    : 136.114.99.52
RemotePort       : 443
InterfaceAlias   : Wi-Fi
SourceAddress    : 192.168.2.178
TcpTestSucceeded : True
```

**What happens**

* TCP 3-way handshake:

  * `SYN`
  * `SYN-ACK`
  * `ACK`

**OSI**

* **L4:** TCP
* **L3:** IP
* **L1/L2:** physical/link underneath

**Good line to remember**

* **TCP handshake proves transport-level connectivity to that port.**

---

### 2) HTTP to your web app

**Command**

```powershell
curl.exe http://136.114.99.52/
```

**What happened**

1. Laptop opened TCP connection to `136.114.99.52:80`
2. TCP handshake completed
3. HTTP request/response happened with Nginx/webserver

**OSI**

* **L7:** HTTP
* **L4:** TCP
* **L3:** IP

**Good line to remember**

* **HTTP worked because TCP connection to port 80 succeeded first, then the HTTP request/response succeeded.**

---

### 3) HTTPS to your web app

**Command**

```powershell
curl.exe -k https://136.114.99.52/
```

**What happened**

1. Laptop opened TCP connection to `136.114.99.52:443`
2. TCP handshake completed
3. **TLS handshake** completed
4. HTTP ran inside TLS, so HTTPS worked

**OSI**

* **L7:** HTTP
* **Between L7 and L4:** TLS
* **L4:** TCP
* **L3:** IP

**Good line to remember**

* **HTTPS = TCP handshake → TLS handshake → HTTP over TLS.**

---

### 4) SSH into the VM on custom port 9090

**Command**

```bash
ssh -p 9090 manojcloudvm@136.114.99.52
```

**What happened**

1. Laptop opened TCP connection to `136.114.99.52:9090`
2. TCP handshake completed
3. SSH protocol handshake happened
4. Authentication happened
5. Remote encrypted shell session was established

**OSI**

* **L7:** SSH
* **L4:** TCP
* **L3:** IP

**Good line to remember**

* **SSH is an application protocol running over TCP, just like HTTP does, but used for secure remote shell access.**

---

### 5) Listening on a port with netcat

**Command on VM**

```bash
nc -lv 9090
```

**What it means**

* VM is listening on TCP port `9090`
* you can use it to observe whether a client can connect to that port

**What it proves**

* useful for testing **raw TCP connectivity**
* does **not** mean HTTP/HTTPS/SSH by itself

---

### 6) Watching the handshake live with tcpdump

**On VM**

```bash
sudo tcpdump -n -i any tcp port 9090
sudo tcpdump -n -i any tcp port 443
```

**What you used it for**

* watching packets in real time while testing from laptop

**What you typically see for a handshake**

* `S` = SYN
* `S.` = SYN-ACK
* `.` = ACK

That confirms the **TCP 3-way handshake**.

**Important**

* `tcpdump` on a public VM may show lots of unrelated internet traffic too, especially on port 443

**Better interpretation**

* use it as packet-level proof that your laptop reached that VM port

---

## Quick mapping by protocol

### HTTP

```powershell
curl.exe http://136.114.99.52/
```

* **Handshake:** TCP handshake
* **Protocol stack:** HTTP → TCP → IP

### HTTPS

```powershell
curl.exe -k https://136.114.99.52/
```

* **Handshakes:** TCP handshake, then TLS handshake
* **Protocol stack:** HTTP → TLS → TCP → IP

### SSH

```bash
ssh -p 9090 manojcloudvm@136.114.99.52
```

* **Handshakes:** TCP handshake, then SSH setup/auth
* **Protocol stack:** SSH → TCP → IP

### Raw TCP test

```bash
nc -lv 9090
sudo tcpdump -n -i any tcp port 9090
```

* **Handshake:** TCP handshake only
* **Protocol stack:** TCP → IP

---

## Best interview summary

**From my Windows PowerShell laptop to the GCP VM, I validated connectivity at multiple layers. For HTTP on port 80, TCP handshake completed first and then HTTP request/response succeeded. For HTTPS on port 443, TCP handshake completed first, then TLS handshake succeeded, and then HTTP was exchanged securely over TLS. For SSH on custom port 9090, TCP handshake completed first, then SSH established an encrypted remote shell session. I also used `tcpdump` on the VM to watch SYN, SYN-ACK, and ACK packets live and confirm the TCP handshakes at the packet level.**

## Tiny memory shortcut

* **HTTP** = TCP + HTTP
* **HTTPS** = TCP + TLS + HTTP
* **SSH** = TCP + SSH
* **tcpdump** = packet-level proof
* **nc -lv** = simple TCP listener/test door

If you want, I can turn this into a **1-page interview cheat sheet**.
